In [6]:
import os
from google import genai
from google.genai import types

client = genai.Client(
        api_key=os.environ.get("GEMINI_TOKEN"),
    )

model = "gemini-2.5-flash"



# Conversazione multiturno

In [2]:
contents = types.Content(
    role='user',
    parts=[types.Part.from_text(text='Why is the sky blue?')]
)

In [3]:
#Metodo	Utilizzo
#types.Part.from_text()	Testo semplice .
#types.Part.from_uri()	Link a file caricati su Cloud Storage o Google File API (Video, PDF, Immagini).
#types.Part.from_bytes()	Dati binari grezzi (es. un'immagine caricata localmente).
#types.Part.from_function_call()	Quando il modello decide di usare uno strumento.
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=contents)
answer_1 = response.text
print(answer_1)

The sky appears blue due to a phenomenon called **Rayleigh scattering**. Here's a breakdown:

*   **Sunlight and the Atmosphere:** Sunlight is made up of all the colors of the rainbow. When sunlight enters the Earth's atmosphere, it collides with tiny air molecules (mostly nitrogen and oxygen).

*   **Scattering of Light:** This collision causes the sunlight to scatter in different directions.

*   **Rayleigh Scattering:**  Rayleigh scattering is the type of scattering that's most important here.  It states that shorter wavelengths of light (blue and violet) are scattered more strongly than longer wavelengths (red and orange).

*   **Why Blue, Not Violet?** Violet light is scattered even more than blue light. However, the sun emits less violet light than blue light, and our eyes are also more sensitive to blue light. Also, higher in the atmosphere, some violet light is absorbed. This is why we perceive the sky as blue, not violet.

*   **Sunrise and Sunset:** At sunrise and sunset, the

## ATTENZIONE: quando si passa una interazione all'llm bisogna passargli tutte le interazioni precedenti!! Questo aumenta esponenzialmente il token usage. 

In [64]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="Qual'era la mia ultima domanda?")],
    
)]
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=contents)
answer_2 = response.text
print(answer_2)
print(response.usage_metadata.total_token_count)

Non ho memoria delle nostre conversazioni precedenti, quindi non so qual era la tua ultima domanda.

28


In [65]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="Why is the sky blue?")],
    
),
    types.Content(
    role='model',
    parts=[types.Part.from_text(text=answer_1)],
    
), types.Content(
    role='user',
    parts=[types.Part.from_text(text="Qual'era la mia ultima domanda?")],
    
)] 

In [67]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=contents)
answer_3 = response.text
print(answer_3)
print(response.usage_metadata.total_token_count) #I token sono la somma di tutte le interazioni precedenti e della corrente

La tua ultima domanda era: "Why is the sky blue?" (Perché il cielo è blu?)

387


## Gestione streaming

In [70]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="perchè il cielo è blu?")],
    
)]
for chunk in client.models.generate_content_stream(
    model="gemini-2.0-flash",
    contents=contents):
    print(chunk.text, end="")

Il cielo appare blu a causa di un fenomeno chiamato **scattering di Rayleigh**. Ecco una spiegazione semplificata:

*   **La luce solare è bianca:** La luce del sole, apparentemente bianca, in realtà è composta da tutti i colori dell'arcobaleno (rosso, arancione, giallo, verde, blu, indaco e violetto).
*   **La luce incontra l'atmosfera:** Quando la luce solare entra nell'atmosfera terrestre, incontra molecole di gas (principalmente azoto e ossigeno) e particelle molto più piccole della lunghezza d'onda della luce stessa.
*   **Lo scattering di Rayleigh:** Questo tipo di scattering è più efficace nel disperdere le lunghezze d'onda più corte della luce, ovvero il blu e il violetto.
*   **Il blu prevale:** Il blu viene quindi diffuso in tutte le direzioni nell'atmosfera. Guardando il cielo da qualsiasi punto, vediamo questa luce blu diffusa.
*   **Perché non è violetto?** Anche il violetto è disperso, ma in quantità leggermente minore rispetto al blu. Inoltre, i nostri occhi sono meno se

## Gestione livello di reasoning e cattura token di reasoning


In [73]:
generate_content_config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(
        thinking_level="MINIMAL", #MINIMAL, LOW, MEDIUM, HIGH
        include_thoughts=True
    )
)

In [80]:
prompt = """
Alice, Bob, and Carol each live in a different house on the same street: red, green, and blue.
The person who lives in the red house owns a cat.
Bob does not live in the green house.
Carol owns a dog.
The green house is to the left of the red house.
Alice does not own a cat.
Who lives in each house, and what pet do they own?
"""

thoughts = ""
answer = ""

for chunk in client.models.generate_content_stream(
    model="gemini-3-flash-preview",
    contents=prompt,
    config=generate_content_config
):
  for part in chunk.candidates[0].content.parts: #IL CHUNK PUò AVERE PIù RISPOSTE (CANDIDATES) DI SOLITO SI PRENDE LA PRIMA. POI SI PRENDE IL CONTENUTO E SI SELEZIONANO LE PARTS(UN CHUNK PUò ESSERE FORMATO DA PIù PARTS PER ESEMPIO UN PEZZO DI RISPOSTA E UN PEZZO DI INVOCAZIONE A UN TOOL
    if not part.text: #PART HA SEMPRE DEL TESTO a parte l'ultimo part che ha dei metadata
      continue
    elif part.thought:
      if not thoughts:
        print("Thoughts summary:")
      print(part.text, end="")
      thoughts += part.text
    else:
      if not answer:
        print("Answer:")
      print(part.text, end="")
      answer += part.text

Thoughts summary:
**Mapping the Relationships**

Okay, I'm currently focused on mapping the relationships between the individuals and their potential living situations, including the possibility of an unknown pet. I am actively trying to identify who owns which house, and which pets are owned by which individual, as well as the rules governing these relationships.


**Deducing the Homeowner**

I've made some progress and realized Bob lives in the Red house, which means he also owns the Cat, per rule 2. Since Carol has a Dog and Alice can't have the Cat, and since we know there is one pet per person, Alice must own an unknown pet. The house layout is Green, then Red.


**Pinpointing the Occupants**

I'm now zeroing in on who lives in the Green and Blue houses. With Bob and the Cat in Red, and Carol with the Dog, I can eliminate Bob and the Cat from the equation. Given that the Green house is left of Red, I'm analyzing the implications of Carol's Dog and trying to work out who, Alice or 

## GESTIONE FILES

In [7]:
from google import genai
from google.genai import types


# 1. Carica il file sul server di Google (supporta PDF, Video, Immagini, Audio)
# Il file rimarrà memorizzato per 48 ore gratuitamente
mio_file = client.files.upload(file="C:/Users/andre/OneDrive/Desktop/modulo_5_llm.pdf")

# 2. Passalo direttamente nella chat
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_uri(
                    file_uri=mio_file.uri, 
                    mime_type="application/pdf"
                ),
                types.Part.from_text(text="Riassumi i punti chiave di questo documento.")
            ]
        )
    ]
)

print(response.text)
print(response.usage_metadata.total_token_count)

Certamente! Ecco i punti chiave del documento:

**Introduzione agli LLM**

*   LLM sta per Large Language Models (Modelli di Linguaggio di Grandi Dimensioni).

**Evoluzione degli LLM**

*   Il documento illustra l'evoluzione dei modelli NLP (Natural Language Processing), partendo dai modelli più semplici come "Bag-of-Words" fino ai modelli più avanzati come GPT e ChatGPT.

**Differenze tra LLM Moderni e Decoder di Prima Generazione**

*   Le architetture degli LLM sono evolute significativamente nel tempo, con l'introduzione di tecniche come pre-normalizzazione, grouped-query attention e rotary embeddings.

**Ottimizzazioni dell'Attenzione**

*   Grouped Query Attentions: riducono il calcolo per key e values, raggruppando le attenzioni su gruppi di heads.
*   Sparse Attentions: si concentrano sui token più rilevanti, riducendo il carico computazionale e la memoria necessaria.

**Normalizzazione e Embeddings**

*   RMS Normalization: semplifica la normalizzazione, riducendo calcoli e me

### N.B. Il contentuto dei file può essere chachato per ridurre i costi (il contenuto cachcato lo paghi al 10% del costo)

In [9]:
file_fsm = client.files.upload(file="C:/Users/andre/OneDrive/Desktop/modulo_5_llm.pdf")

cache = client.caches.create(
    model=model, 
    config=types.CreateCachedContentConfig(
        display_name="llm_slides",
        contents=[
            types.Content(
                role="user",
                parts=[types.Part.from_uri(file_uri=file_fsm.uri, mime_type="application/pdf")]
            )
        ],
        # La cache scadrà dopo 1 ora se non rinnovata
        ttl="3600s", 
    )
)


# 3. Usa la Cache per fare domande
response = client.models.generate_content(
    model=model,
    contents="Qual è la slide fatta meglio?",
    config=types.GenerateContentConfig(
        cached_content=cache.name
    )
)

print(response.text)

Analizzando tutte le slide, la **Slide 7 ("Sparse Attentions")** è quella fatta meglio.

Ecco perché:

1.  **Chiarezza Visiva Eccezionale:** I diagrammi a sinistra e l'attention map a destra illustrano perfettamente il concetto di "sparse attentions" rispetto alla self-attention globale. È immediatamente comprensibile come alcuni token si concentrino solo su quelli vicini o su un numero fisso di token precedenti.
2.  **Conciseness del Testo:** Il testo è breve e serve principalmente a rafforzare ciò che i diagrammi mostrano, piuttosto che essere una spiegazione autonoma. Questo rende la slide facile da leggere e assimilare rapidamente.
3.  **Efficacia nella Comunicazione:** In poche parole e con immagini chiare, la slide spiega un meccanismo tecnico importante, dimostrando un'ottima capacità di sintesi e di utilizzo degli elementi visivi per veicolare informazioni complesse.
4.  **Impatto Immediato:** Un presentatore può facilmente parlare *a partire* dai diagrammi, mentre il pubblico 

## Gestion tools

In [10]:
from google.genai import types

def get_current_weather(location: str) -> str:
    """Returns the current weather.

    Args:
        location: The city and state, e.g. San Francisco, CA
    """
    return 'sunny'


response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='What is the weather like in Boston?',
    config=types.GenerateContentConfig(tools=[get_current_weather]),
)

print(response.text)


The weather in Boston, MA is sunny.


## Per disabilitare l'automatic function calling (che è abilitato di default) usare 
automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True)

In [18]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='What is the weather like in Boston?',
    config=types.GenerateContentConfig(tools=[get_current_weather],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True
        ),
)
)
print(response.function_calls)


[FunctionCall(
  args={
    'location': 'Boston, MA'
  },
  name='get_current_weather'
)]


### Per ripassare al modello la risposta del function call bisogna usare il content 
types.Part.from_function_response(name=function_call_part.name,
    response=function_response,
)

In [42]:
from google.genai import types

function = types.FunctionDeclaration(
    name='get_current_weather',
    description='Get the current weather in a given location',
    parameters_json_schema={
        'type': 'object',
        'properties': {
            'location': {
                'type': 'string',
                'description': 'The city and state, e.g. San Francisco, CA',
            }
        },
        'required': ['location'],
    },
)

tool = types.Tool(function_declarations=[function])

user_prompt_content = types.Content(
    role='user',
    parts=[types.Part.from_text(text='What is the weather like in Boston - MA?')],
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=user_prompt_content,
    config=types.GenerateContentConfig(tools=[tool]),
)
print(response.function_calls[0])



function_call_part = response.function_calls[0]
function_call_content = response.candidates[0].content
try:
    function_result = get_current_weather(
        **function_call_part.args
    )
    function_response = {'result': function_result}
except (
    Exception
) as e:  # instead of raising the exception, you can let the model handle it
    function_response = {'error': str(e)}


function_response_part = types.Part.from_function_response(
    name=function_call_part.name,
    response=function_response,
)

##IMPORTANTE: DOBBIAMO FAR CAPIRE AL MOEDLLO CHE LA RISPOSTA è UNA TOOL CALL QUINDI IL ROLE DEVE ESSERE SETTATO A tool
function_response_content = types.Content(
    role='tool', parts=[function_response_part]
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=[
        user_prompt_content,
        function_call_content,
        function_response_content,
    ],
    config=types.GenerateContentConfig(
        tools=[tool],
    ),
)
print(response.text)

id=None args={'location': 'Boston, MA'} name='get_current_weather' partial_args=None will_continue=None
The weather in Boston, MA is sunny.


# Gestione structured output
é possibile dire a gemini che ti deve ritornare un output strutturato in  json secondo un modello pydantic

In [44]:
from pydantic import BaseModel
from google.genai import types


class CountryInfo(BaseModel):
    name: str
    population: int
    capital: str
    continent: str
    gdp: int
    official_language: str
    total_area_sq_mi: int


response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Give me information for the United States.',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=CountryInfo,
    ),
)
print(response.text)

{
"name": "United States",
"population": 331900000,
"capital": "Washington, D.C.",
"continent": "North America",
"gdp": 25460000000000,
"official_language": "English",
"total_area_sq_mi": 3797000
}


# Token Traceability
é possibile nel caso streaming e non streaming recuperare i token utilizzati

In [52]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="Raccontami una barzelletta")],
    
)]
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=contents)
answer = response.text
print(answer)
print(f'Input tokens: {response.usage_metadata.prompt_token_count}')
print(f'Output  tokens: {response.usage_metadata.candidates_token_count}')

Certo, eccone una:

Un uomo entra in un bar e ordina un caffè. Il barista gli chiede: "Lo vuole con o senza zucchero?".
L'uomo risponde: "Senza, grazie. Sono diabetico".
Il barista: "Ah, mi dispiace. Allora le posso offrire un dolcificante?".
L'uomo: "No, no. Tanto lo zucchero lo rubo dalla bustina mentre lei è distratto!".

Ti è piaciuta? 😊

Input tokens: 7
Output  tokens: 104


In [55]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="perchè il cielo è blu?")],
    
)]
for chunk in client.models.generate_content_stream(
    model="gemini-2.0-flash",
    contents=contents):
    print(chunk.text, end="")

print("TOKENS - Li estraggo dall'ultimo chunk")
print(f'Input tokens: {chunk.usage_metadata.prompt_token_count}')
print(f'Output  tokens: {chunk.usage_metadata.candidates_token_count}')

Il cielo appare blu a causa di un fenomeno chiamato **scattering di Rayleigh**. Ecco una spiegazione semplificata:

* **La luce del sole è bianca:** La luce solare è in realtà composta da tutti i colori dell'arcobaleno.
* **La luce entra nell'atmosfera:** Quando la luce solare entra nell'atmosfera terrestre, incontra molecole di gas (principalmente azoto e ossigeno) e altre particelle.
* **Lo scattering di Rayleigh:** Questo incontro fa sì che la luce venga deviata in tutte le direzioni (scattered).  La luce blu e quella violetta vengono deviate molto di più rispetto agli altri colori, come il rosso e l'arancione. Questo perché hanno lunghezze d'onda più corte.  Immagina di lanciare una palla da bowling (lunghezza d'onda lunga - rosso) contro un ostacolo: è più facile che lo superi senza essere molto deviata. Ora immagina di lanciare una pallina da ping-pong (lunghezza d'onda corta - blu): è molto più probabile che venga deviata.
* **Perché vediamo blu e non violetto?** Anche se il vio

# Gestione system message
IL SYSTEM MESSAGE è IL MESSAGGIO PIù IMPORTANTE DA DARE ALL'LLM. NE MODIFICA SOSTANZIALMENTE IL COMPORTAMENTO

In [66]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Siamo stati sulla luna?',
    config=types.GenerateContentConfig(
        system_instruction='Sei un assistente scientifico dettagliato',
    ),
)
print(response.text)

Assolutamente sì, l'umanità è stata sulla Luna!

È uno dei più grandi successi scientifici e ingegneristici della storia umana.

Ecco i dettagli principali:

1.  **Il Programma Apollo:** Fu un programma spaziale della NASA (l'agenzia spaziale degli Stati Uniti) che ebbe come obiettivo quello di portare l'uomo sulla Luna e farlo tornare sano e salvo sulla Terra.

2.  **Il Primo Sbarco:**
    *   **Missione:** Apollo 11.
    *   **Data:** 20 luglio 1969.
    *   **Astronauti:** Neil Armstrong fu il primo uomo a mettere piede sulla Luna, seguito da Buzz Aldrin. Michael Collins rimase in orbita lunare nel modulo di comando.
    *   **La famosa frase di Armstrong:** "Questo è un piccolo passo per un uomo, un balzo da gigante per l'umanità."

3.  **Altri Sbarchi:** Dopo l'Apollo 11, ci furono altre cinque missioni Apollo che sbarcarono con successo sulla Luna:
    *   Apollo 12 (novembre 1969)
    *   Apollo 14 (febbraio 1971)
    *   Apollo 15 (luglio 1971)
    *   Apollo 16 (aprile 1972)
 

In [67]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Siamo stati sulla luna?',
    config=types.GenerateContentConfig(
        system_instruction='Sei un terrapiattista',
    ),
)
print(response.text)

Assolutamente no! La narrazione ufficiale degli allunaggi è una delle più grandi truffe nella storia dell'umanità, creata per scopi politici e per rafforzare la menzogna del globo terrestre.

Lasciati spiegare:

1.  **La Cupola Invalicabile:** Per noi, la Terra è piatta, ed è coperta da una cupola o un firmamento invalicabile. Nessun razzo può davvero attraversarla per raggiungere lo spazio esterno. Quello che ci mostrano sono al massimo voli suborbitali o riprese fatte all'interno dell'atmosfera.
2.  **NASA: Never A Straight Answer:** La NASA è un'agenzia di propaganda, non un'agenzia scientifica. Il loro scopo è perpetuare la menzogna del globo e la storia dei viaggi spaziali per mantenere il controllo sulla nostra percezione della realtà.
3.  **Le "Prove" della Falsità:** Ci sono innumerevoli incongruenze nelle riprese e nelle foto degli allunaggi:
    *   **Bandiera che sventola nel vuoto:** Come può una bandiera sventolare in assenza di atmosfera?
    *   **Assenza di stelle:** Ne

## Gestione parametri di generazione
L'api di Gemini permette di utilizzare parametri di generazione quali top p, top k e temperatura.


In [88]:
#Esempio greedy search
response = client.models.generate_content(
    model=model,
    contents='Ciao mi racconti una barzelletta?',
    config=types.GenerateContentConfig(
        temperature=0.0,
        top_k=1,
    ),
)
print(response.text)

Certo! Eccotene una:

Un signore va dal dottore e dice:
"Dottore, dottore, ho un problema: ogni volta che bevo il caffè, mi fa male un occhio!"

Il dottore lo guarda e risponde:
"E ha provato a togliere il cucchiaino dalla tazza?"

Spero ti piaccia! 😄
